### Process End-of-Semester Survey Responses

In [ ]:
import csv
import os
import re
from collections import defaultdict, Counter
from reportlab.lib.pagesizes import LETTER
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch

def get_leader_name(row):
    for col, value in row.items():
        if col == "Text" and value and value.strip() and "-" in value:
            return value.strip()
    return "N/A"

def split_leader_and_time(raw_value):
    if "MATH" in raw_value:
        name, time = raw_value.split("MATH", 9)
        return name.strip(), time.strip()
    elif "CHEM" in raw_value:
        name, time = raw_value.split("CHEM", 9)
        return name.strip(), time.strip()
    return raw_value.strip(), "Unspecified Time"

def safe_name(name):
    return "".join(
        c for c in name
        if c.isalnum() or c in " .-_:"
    ).strip()

def normalize_name(name):
    name = name.strip()
    # Remove multiple spaces
    name = re.sub(r'\s+', ' ', name)
    # Add space between lowercase and uppercase letters (e.g., 'RiyaT' -> 'Riya T')
    name = re.sub(r'([a-z])([A-Z])', r'\1 \2', name)
    # Add space between letters and numbers (e.g., 'Monday10:30AM' -> 'Monday - 10:30AM')
    name = re.sub(r'([A-Za-z])(\d)', r'\1 - \2', name)
    # Ensure AM/PM has proper spacing
    name = re.sub(r'(\d)(AM|PM)', r'\1 \2', name)
    # Add separator between consecutive times
    # Example: 10:30 AM11:50 AM -> 10:30 AM - 11:50 AM
    name = re.sub(
        r'((?:\d{1,2}:\d{2}\s?(?:AM|PM)))((?:\d{1,2}:\d{2}\s?(?:AM|PM)))',
        r'\1 - \2',
        name
    )
    return name

parent_dir = os.getcwd()
data_dir = os.path.join(parent_dir, 'data')
csv_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.lower().endswith('.csv')]

subjects = set()
subject_files = {}
selected_range = (17, 39)
excluded_columns = {'How many PLTL sessions did you attend this semester?'}
reports = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: {'ratings': defaultdict(list), 'feedback': defaultdict(list)})))

for csv_filename in csv_files:
    print(f"Processing file: {csv_filename}")
    with open(csv_filename, newline='', encoding='utf-8') as formalcsv:
        reader = csv.reader(formalcsv)
        headers = next(reader)
        selected_columns = [
            col for col in headers[selected_range[0]:selected_range[1]] if col not in excluded_columns
        ]

        for row in reader:
            subject_index = headers.index('What subject are you enrolled in for PLTL support?')
            subject = row[subject_index]

            # Get employee name from any of the last 11 columns
            employee_raw = None
            for val in row[-11:]:
                if val.strip() and "-" in val:
                    employee_raw = val.strip()
                    break
            if not employee_raw:
                for val in row[-11:]:
                    if val.strip():
                        employee_raw = val.strip()
                        break
            if not employee_raw:
                continue

            #print("Row employee_raw:", employee_raw)
            employee_name_raw, day_time = split_leader_and_time(employee_raw)
            employee_name = safe_name(employee_name_raw)
            employee_name = normalize_name(employee_name)
            day_time = safe_name(day_time)
            day_time = day_time[4:]
            day_time = normalize_name(day_time)

            for i, col_name in enumerate(headers):
                if col_name in selected_columns:
                    value = row[i].strip()
                    if not value:
                        continue
                    if value:
                        # Text feedback
                        reports[subject][employee_name][day_time]['feedback'][col_name].append(value)

for subject, employees in reports.items():
    subject_dir = os.path.join(parent_dir, safe_name(subject))
    os.makedirs(subject_dir, exist_ok=True)

    for employee, schedules in employees.items():
        pdf_path = os.path.join(subject_dir, f"{employee}.pdf")
        print(pdf_path)
        
        doc = SimpleDocTemplate(pdf_path, pagesize=LETTER,
                                rightMargin=inch, leftMargin=inch,
                                topMargin=inch, bottomMargin=inch)
        elements = []
        styles = getSampleStyleSheet()
        
        # Title
        elements.append(Paragraph(f"<b>{employee}</b>", styles['Title']))
        elements.append(Spacer(1, 0.2*inch))
        
        for day_time, data in schedules.items():
            num_responses = 0
            # Count the maximum number of entries in any rating question (they all should have the same number)
            if data['ratings']:
                num_responses = max(len(answers) for answers in data['ratings'].values())
            elif data['feedback']:
                num_responses = max(len(responses) for responses in data['feedback'].values())

            elements.append(Paragraph(f"<b>Session: {day_time} (Responses: {num_responses})</b>", styles['Heading3']))
            elements.append(Spacer(1, 0.1*inch))
            
            # Get answers for each question
            elements.append(Spacer(1, 0.1*inch))
            excluded_questions = {'What subject are you enrolled in for PLTL support?', 'Text'}
            counted_questions = {
                "How did your grade change after participating in PLTL?",
                "How much did PLTL contribute to your success in the course?",
                "What was the MOST valuable part of PLTL?",
                "What improved the MOST for you because of PLTL?",
            }
            # Feedback as bulleted list
            for question, responses in data['feedback'].items():
                # Exclude N/A or n/a responses
                responses = [
                    r for r in responses
                    if r.strip().lower() not in {"n/a", "na", "none"}
                ]
                if question not in excluded_questions and responses:
                    elements.append(Paragraph(f"<b>{question}:</b>", styles['Normal']))
                    if question in counted_questions:
                        response_counts = Counter(responses)
                        for answer, count in response_counts.items():
                            elements.append(Paragraph(f"{count} student(s): {answer}", styles['Normal']))
                            elements.append(Spacer(1, 0.05*inch))
                        elements.append(Spacer(1, 0.2*inch))
                    else:
                        for response in responses:
                            elements.append(Paragraph(f"• {response}", styles['Normal']))
                        elements.append(Spacer(1, 0.2*inch))
        
        doc.build(elements)
        print(f"Created PDF: {pdf_path}")

Processing file: c:\Users\ammil\Documents\College\Observations\S26 - End of Semester\data\Spring 2026 End-of-Semester Survey_May 26, 2026_12.57.csv
c:\Users\ammil\Documents\College\Observations\S26 - End of Semester\MATH 2418\Swapnil Singh.pdf
Created PDF: c:\Users\ammil\Documents\College\Observations\S26 - End of Semester\MATH 2418\Swapnil Singh.pdf
c:\Users\ammil\Documents\College\Observations\S26 - End of Semester\MATH 2418\Nicholas Grudzinski.pdf
Created PDF: c:\Users\ammil\Documents\College\Observations\S26 - End of Semester\MATH 2418\Nicholas Grudzinski.pdf
c:\Users\ammil\Documents\College\Observations\S26 - End of Semester\MATH 2418\Ishaan Gupta.pdf
Created PDF: c:\Users\ammil\Documents\College\Observations\S26 - End of Semester\MATH 2418\Ishaan Gupta.pdf
c:\Users\ammil\Documents\College\Observations\S26 - End of Semester\MATH 2418\Ela Guo.pdf
Created PDF: c:\Users\ammil\Documents\College\Observations\S26 - End of Semester\MATH 2418\Ela Guo.pdf
c:\Users\ammil\Documents\College\O